# Medical FHE Walkthrough with TenSEAL

Notebook này đi qua toàn bộ pipeline: đọc ảnh X-ray mẫu, trích xuất feature vector, mã hóa bằng CKKS qua TenSEAL, chạy linear score trên ciphertext, rồi so sánh kết quả encrypted inference với plaintext.

> Đây là prototype giáo dục, không phải mô hình chẩn đoán y tế.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'scripts').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from scripts.medical_fhe_pipeline import (
    DEFAULT_DATA_DIR,
    DEFAULT_MODEL_PATH,
    create_ckks_context,
    extract_xray_features,
    list_image_paths,
    load_grayscale_image,
    load_triage_model,
    plaintext_linear_logit,
    sigmoid,
    ts,
)

image_paths = list_image_paths(DEFAULT_DATA_DIR)
model = load_triage_model(DEFAULT_MODEL_PATH)
len(image_paths), model.name

## 1. Client giữ ảnh X-ray local

Trong kiến trúc FHE, ảnh gốc không cần rời khỏi phía bệnh viện/client. Notebook chỉ dùng ảnh mẫu trong thư mục `data/chestxray_sample/`.

In [ ]:
image_path = image_paths[0]
image = load_grayscale_image(image_path)

plt.figure(figsize=(4, 4))
plt.imshow(image, cmap='gray')
plt.title(image_path.name)
plt.axis('off');

## 2. Trích xuất feature vector

Demo dùng 10 đặc trưng ảnh nhỏ để minh họa encrypted inference. Với nghiên cứu thật, phần này có thể thay bằng feature extractor hoặc model đã được thiết kế lại cho FHE.

In [ ]:
features = extract_xray_features(image)
for name, value in zip(model.feature_names, features):
    print(f'{name:18} {value:.6f}')

## 3. Plaintext baseline

Ta tính score plaintext để có baseline so sánh sai số CKKS.

In [ ]:
plain_logit = plaintext_linear_logit(features, model)
plain_probability = sigmoid(plain_logit)
plain_logit, plain_probability

## 4. TenSEAL CKKS encrypted inference

Server chỉ nhận `Enc(x)` và tính `Enc(x) dot w + b`. Secret key nằm trong context phía client trong demo này; trong triển khai thật cần tách rõ public/evaluation keys và secret key.

In [ ]:
if ts is None:
    raise RuntimeError('TenSEAL is not installed in this environment.')

context = create_ckks_context()
encrypted_features = ts.ckks_vector(context, features.tolist())
encrypted_logit_vector = encrypted_features.dot(model.weights) + model.bias
encrypted_logit = float(encrypted_logit_vector.decrypt()[0])
encrypted_probability = sigmoid(encrypted_logit)

print(f'Plaintext probability: {plain_probability:.8f}')
print(f'Encrypted probability: {encrypted_probability:.8f}')
print(f'Absolute logit error: {abs(encrypted_logit - plain_logit):.2e}')

## 5. Batch demo trên tất cả ảnh mẫu

In [ ]:
rows = []
for path in image_paths:
    arr = load_grayscale_image(path)
    x = extract_xray_features(arr)
    p_logit = plaintext_linear_logit(x, model)
    enc = ts.ckks_vector(context, x.tolist())
    e_logit = float((enc.dot(model.weights) + model.bias).decrypt()[0])
    rows.append((path.name, sigmoid(p_logit), sigmoid(e_logit), abs(e_logit - p_logit)))

for name, p_plain, p_enc, err in rows:
    print(f'{name:14} plain={p_plain:.4f} encrypted={p_enc:.4f} err={err:.2e}')

## Kết luận

Demo cho thấy linear scoring có thể chạy trực tiếp trên ciphertext với sai số nhỏ. Giới hạn hiện tại là feature extraction vẫn ở client và model chỉ là tuyến tính; để mở rộng có thể benchmark nhiều tham số CKKS, train model với nhãn thật, hoặc dùng polynomial activations cho shallow neural network.